# 01 - Preprocessing: Dokumen, Query, dan Penilaian relevansi siap pakai

Alur: muat data interim → bentuk Dokumen dan periksa kunci → buang qrels yang id-nya tidak ada → periksa setiap query punya dokumen relevan → bagi query val/test 50:50 seed 42 → hitung hash isi dan bandingkan dengan hasil lama → simpan ke `data/processed/`.

Keluaran di `data/processed/`:
- `documents.parquet` — kolom `doc_id`, `title`, `text`
- `queries.parquet` — kolom `query_id`, `text`, `split`
- `qrels.parquet` — kolom `query_id`, `doc_id`, `relevance`
- `metadata.json` — sumber dan revision dataset, aturan split, jumlah baris, jumlah qrels yang dibuang, hash isi

Prasyarat: `00_load_dataset` sudah dijalankan.

Keputusan: K2 (001a, 002a), D3, model data (002a), K9 (003a).

In [ ]:
from pathlib import Path

# Keluaran 00_load_dataset: corpus/topics/qrels.parquet dan metadata.json
INTERIM_DIR = Path("../data/interim")
# Folder keluaran notebook ini, dibaca 02_embedding dan notebook penilai
PROCESSED_DIR = Path("../data/processed")

# Seed pembagian query val/test (K2)
RANDOM_SEED = 42
# Porsi query untuk split val; sisanya test (K2: 50:50)
VAL_FRACTION = 0.5
# Dokumen relevan kalau relevance di qrels paling kecil nilai ini (002a)
MIN_RELEVANCE = 1

# Kolom tiap tabel keluaran, juga urutan kolom saat menghitung hash isi
DOCUMENT_COLUMNS = ["doc_id", "title", "text"]
QUERY_COLUMNS = ["query_id", "text", "split"]
QRELS_COLUMNS = ["query_id", "doc_id", "relevance"]
# Kolom yang dikunci hash pembagian query (K2)
SPLIT_COLUMNS = ["query_id", "split"]

## 1. Muat data interim

In [ ]:
import json

import pandas as pd


def load_parquet(path: Path) -> pd.DataFrame:
    """Baca satu tabel Parquet.

    Args:
        path: Berkas Parquet.

    Returns:
        Isi tabel.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {path}")
    return pd.read_parquet(path)


def load_json(path: Path) -> dict[str, object]:
    """Baca satu berkas JSON.

    Args:
        path: Berkas JSON.

    Returns:
        Isi berkas.

    Raises:
        FileNotFoundError: Berkas tidak ada.
    """
    if not path.is_file():
        raise FileNotFoundError(f"Berkas tidak ditemukan: {path}")
    with path.open(encoding="utf-8") as file:
        return json.load(file)


corpus = load_parquet(INTERIM_DIR / "corpus.parquet")
topics = load_parquet(INTERIM_DIR / "topics.parquet")
raw_qrels = load_parquet(INTERIM_DIR / "qrels.parquet")
interim_metadata = load_json(INTERIM_DIR / "metadata.json")
print({"corpus": len(corpus), "topics": len(topics), "qrels": len(raw_qrels)})

## 2. Dokumen

`docid` MIRACL menjadi `doc_id`. doc_id wajib terisi dan unik (model data 002a); isi dokumen tidak diubah.

In [ ]:
def to_documents(corpus: pd.DataFrame) -> pd.DataFrame:
    """Ubah tabel korpus menjadi tabel Dokumen.

    Args:
        corpus: Korpus interim dengan kolom docid, title, text.

    Returns:
        Tabel Dokumen dengan kolom doc_id, title, text dalam urutan baris korpus.
    """
    return corpus.rename(columns={"docid": "doc_id"})[["doc_id", "title", "text"]].reset_index(drop=True)


def validate_unique_keys(frame: pd.DataFrame, key_columns: list[str], name: str) -> None:
    """Pastikan kolom kunci terisi dan tidak ganda.

    Args:
        frame: Tabel yang diperiksa.
        key_columns: Kolom yang membentuk kunci.
        name: Nama tabel untuk pesan error.

    Raises:
        ValueError: Ada kunci kosong atau ganda.
    """
    empty_rows = int(frame[key_columns].isna().any(axis=1).sum())
    if empty_rows:
        raise ValueError(f"{name}: {empty_rows} baris dengan kunci kosong di {key_columns}")
    duplicate_rows = int(frame.duplicated(subset=key_columns).sum())
    if duplicate_rows:
        raise ValueError(f"{name}: {duplicate_rows} baris dengan kunci ganda di {key_columns}")


documents = to_documents(corpus)
validate_unique_keys(documents, ["doc_id"], "documents")
print({"documents": len(documents)})

## 3. Penilaian relevansi

Penilaian yang query_id atau doc_id-nya tidak ada di topics atau korpus dibuang, dan jumlahnya dicatat (model data 002a). Kolom iterasi TREC (`Q0`) tidak dipakai.

In [ ]:
def filter_qrels(qrels: pd.DataFrame, query_ids: pd.Series, doc_ids: pd.Series) -> tuple[pd.DataFrame, int]:
    """Buang penilaian yang merujuk query atau dokumen yang tidak ada.

    Args:
        qrels: Qrels interim dengan kolom query_id, iteration, doc_id, relevance.
        query_ids: query_id yang ada di topics.
        doc_ids: doc_id yang ada di Dokumen.

    Returns:
        Tabel Penilaian relevansi (query_id, doc_id, relevance) dan jumlah baris yang dibuang.
    """
    known = qrels["query_id"].isin(query_ids) & qrels["doc_id"].isin(doc_ids)
    kept = qrels.loc[known, ["query_id", "doc_id", "relevance"]].reset_index(drop=True)
    return kept, int((~known).sum())


qrels, dropped_qrels = filter_qrels(raw_qrels, topics["query_id"], documents["doc_id"])
validate_unique_keys(qrels, ["query_id", "doc_id"], "qrels")
print({"qrels": len(qrels), "dropped_qrels": dropped_qrels})

## 4. Query dan pembagian val/test

Setiap query wajib punya paling sedikit satu dokumen dengan relevance ≥ 1 (model data 002a). Query diurutkan menurut query_id lalu diacak dengan `RandomState(42)`; separuh pertama urutan acak menjadi val. `RandomState` dipakai karena NumPy menjamin urutan acaknya sama di semua versi, sedangkan `default_rng` tidak; split yang berubah membuat gate checksum di tahap 5 berhenti.

In [ ]:
import numpy as np


def validate_queries_have_relevant(queries: pd.DataFrame, qrels: pd.DataFrame, min_relevance: int) -> None:
    """Pastikan setiap query punya dokumen relevan.

    Args:
        queries: Tabel query dengan kolom query_id.
        qrels: Tabel Penilaian relevansi.
        min_relevance: Nilai relevance terkecil yang dihitung relevan.

    Raises:
        ValueError: Ada query tanpa dokumen relevan.
    """
    relevant_query_ids = qrels.loc[qrels["relevance"] >= min_relevance, "query_id"]
    missing = queries.loc[~queries["query_id"].isin(relevant_query_ids), "query_id"]
    if len(missing):
        raise ValueError(
            f"{len(missing)} query tanpa dokumen relevance >= {min_relevance}, contoh: {missing.head(5).tolist()}"
        )


def split_queries(queries: pd.DataFrame, seed: int, val_fraction: float) -> pd.DataFrame:
    """Bagi query menjadi val dan test secara acak dengan seed tetap.

    Args:
        queries: Tabel query dengan kolom query_id dan text.
        seed: Seed RandomState.
        val_fraction: Porsi query untuk val.

    Returns:
        Tabel query terurut menurut query_id dengan kolom split bernilai "val" atau "test".
    """
    ordered = queries.sort_values("query_id", kind="stable").reset_index(drop=True)
    order = np.random.RandomState(seed).permutation(len(ordered))
    val_count = int(round(len(ordered) * val_fraction))
    split = np.full(len(ordered), "test", dtype=object)
    split[order[:val_count]] = "val"
    return ordered.assign(split=split)


validate_unique_keys(topics, ["query_id"], "topics")
validate_queries_have_relevant(topics, qrels, MIN_RELEVANCE)
queries = split_queries(topics[["query_id", "text"]], RANDOM_SEED, VAL_FRACTION)
print(queries["split"].value_counts().to_dict())

## 5. Hash isi dan gate checksum

Hash dihitung dari isi kanonik (baris diurutkan menurut kolom, setiap baris diserialisasi sebagai JSON), bukan dari byte berkas. Kalau `data/processed/metadata.json` sudah ada dan hash-nya berbeda, notebook berhenti: split atau data yang berubah membuat embedding dan catatan run lama kehilangan pasangannya.

In [ ]:
import hashlib


def compute_content_hash(frame: pd.DataFrame, columns: list[str]) -> str:
    """Hitung SHA-256 dari isi kanonik tabel.

    Args:
        frame: Tabel yang di-hash.
        columns: Kolom yang ikut di-hash, sesuai urutan ini; baris diurutkan menurut kolom yang sama.

    Returns:
        Hash heksadesimal 64 karakter.
    """
    digest = hashlib.sha256()
    ordered = frame[columns].sort_values(columns, kind="stable")
    for row in ordered.itertuples(index=False, name=None):
        digest.update(json.dumps(row, ensure_ascii=False, separators=(",", ":"), default=str).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()


def validate_hashes_unchanged(previous: dict[str, str] | None, current: dict[str, str]) -> None:
    """Pastikan hash isi sama dengan hasil preprocessing sebelumnya.

    Args:
        previous: Hash dari metadata.json lama, atau None kalau belum pernah dibuat.
        current: Hash hasil notebook ini.

    Raises:
        ValueError: Ada hash yang berbeda.
    """
    if previous is None:
        return
    changed = sorted(name for name in current if previous.get(name) != current[name])
    if changed:
        raise ValueError(f"Isi berubah dibanding data/processed yang ada: {changed}; hasil lama tidak ditimpa")


content_hashes = {
    "documents": compute_content_hash(documents, DOCUMENT_COLUMNS),
    "queries": compute_content_hash(queries, QUERY_COLUMNS),
    "qrels": compute_content_hash(qrels, QRELS_COLUMNS),
    "split": compute_content_hash(queries, SPLIT_COLUMNS),
}
processed_metadata_path = PROCESSED_DIR / "metadata.json"
previous_hashes = load_json(processed_metadata_path)["hashes"] if processed_metadata_path.is_file() else None
validate_hashes_unchanged(previous_hashes, content_hashes)
print(content_hashes)

## 6. Simpan ke `data/processed/`

Tabel disimpan sebagai Parquet dan metadata sebagai JSON (K9). Revision dataset dari `data/interim/metadata.json` ikut dibawa supaya bisa dicatat di Set embedding.

In [ ]:
from datetime import datetime, timezone


def build_processed_metadata(
    sources: object, counts: dict[str, int], dropped_qrels: int, hashes: dict[str, str]
) -> dict[str, object]:
    """Susun isi metadata.json data processed.

    Args:
        sources: Bagian "tables" metadata interim (repo, revision, berkas, jumlah baris).
        counts: Jumlah baris tiap tabel dan tiap split.
        dropped_qrels: Jumlah qrels yang dibuang.
        hashes: Hash isi tiap tabel dan split.

    Returns:
        Metadata dengan cap waktu pembuatan.
    """
    # Cap waktu UTC format ISO 8601, contoh 2026-10-02T08:30:00+00:00
    created_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    split_rule = {"seed": RANDOM_SEED, "val_fraction": VAL_FRACTION, "method": "numpy RandomState permutation"}
    return {
        "created_at": created_at,
        "sources": sources,
        "split_rule": split_rule,
        "min_relevance": MIN_RELEVANCE,
        "counts": counts,
        "dropped_qrels": dropped_qrels,
        "hashes": hashes,
    }


def save_processed(tables: dict[str, pd.DataFrame], metadata: dict[str, object], processed_dir: Path) -> None:
    """Simpan tabel sebagai Parquet dan metadata sebagai JSON.

    Args:
        tables: Nama tabel ke isinya; disimpan sebagai `<nama>.parquet` tanpa index.
        metadata: Isi metadata.json.
        processed_dir: Folder tujuan.
    """
    processed_dir.mkdir(parents=True, exist_ok=True)
    for name, table in tables.items():
        table.to_parquet(processed_dir / f"{name}.parquet", index=False)
    with (processed_dir / "metadata.json").open("w", encoding="utf-8", newline="\n") as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)


counts = {
    "documents": len(documents),
    "queries": len(queries),
    "qrels": len(qrels),
    "val": int((queries["split"] == "val").sum()),
    "test": int((queries["split"] == "test").sum()),
}
processed_metadata = build_processed_metadata(interim_metadata["tables"], counts, dropped_qrels, content_hashes)
save_processed(
    {
        "documents": documents[DOCUMENT_COLUMNS],
        "queries": queries[QUERY_COLUMNS],
        "qrels": qrels[QRELS_COLUMNS],
    },
    processed_metadata,
    PROCESSED_DIR,
)
print(counts)